In [2]:
!pip install moviepy requests matplotlib

In [3]:
import requests
from moviepy.editor import VideoFileClip, CompositeVideoClip, vfx
import tempfile
import os
import subprocess
import imageio_ffmpeg
import matplotlib.pyplot as plt
import numpy as np
import math
import re
from IPython.display import Video, display

def get_mp4_url(url):
    """Scrapes the Baseball Savant page to find the raw .mp4 link"""
    # If the provided URL is already a direct video link, return it immediately
    if url.endswith(".mp4"):
        return url

    # Disguise the request to prevent the server from blocking the script
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        raise ValueError(f"Could not load webpage. Status code: {response.status_code}")

    html_text = response.text

    #Looks for sporty-clips video link
    mlb_video_link = r'https://sporty-clips\.mlb\.com/[^"\']+\.mp4'
    mlb_match = re.search(mlb_video_link, html_text)
    if mlb_match:
        return mlb_match.group(0)

    # Otherwise looks for any .mp4
    generic_pattern = r'https://[^"\']+\.mp4'
    generic_match = re.search(generic_pattern, html_text)
    if generic_match:
        return generic_match.group(0)
    # If no links are found return an error
    raise ValueError("Could not find any MP4 links on this webpage.")

def download_video(url, suffix=".mp4"):
    """Downloads the raw video to a temporary file."""
    response = requests.get(url)
    temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)
    temp_file.write(response.content)
    temp_file.close()

    return temp_file.name

def normalize_video(input_path):
    """Standardizes video framerates to 60fps to prevent desync issues during overlay.
    Uses FFmpeg directly because MoviePy's set_fps() struggles with variable framerates."""
    ffmpeg_exe = imageio_ffmpeg.get_ffmpeg_exe()
    output_path = tempfile.mktemp(suffix="_clean.mp4")
    #Uses FFmpeg commands:
    # -r 60 to forcing exactly 60 frames per second
    # -an  to strip audio
    # -c:v libx264 -preset ultrafast to encode the video quickly at the cost of slight file size increases
    command = [
        ffmpeg_exe, "-y", "-i", input_path,
        "-r", "60", "-an",
        "-c:v", "libx264", "-preset", "ultrafast",
        output_path
    ]
    # Run the command silently
    subprocess.run(command, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return output_path


def create_overlay(url1, release_time1, url2, release_time2, speed):
    """Generates a synced, transparent overlay using specific release timestamps."""
    file1_raw, file2_raw = None, None
    file1_clean, file2_clean = None, None
    clip1, clip2 = None, None

    try:
        print("Getting Urls...")
        mp4_url1 = get_mp4_url(url1)
        mp4_url2 = get_mp4_url(url2)

        file1_raw = download_video(mp4_url1)
        file2_raw = download_video(mp4_url2)

        print("Normalizing frame rates...")
        file1_clean = normalize_video(file1_raw)
        file2_clean = normalize_video(file2_raw)

        # Load the cleaned videos into MoviePy objects
        clip1 = VideoFileClip(file1_clean)
        clip2 = VideoFileClip(file2_clean)

        # Make the second clip semi-transparent for the ghosting effect
        clip2 = clip2.set_opacity(0.5)

        print("Syncing release points...")
        # Trim the beginning of the clip with the earlier release time so both release points occur at the exact same frame
        if release_time1 > release_time2:
            trim_amount = release_time1 - release_time2
            clip1 = clip1.subclip(trim_amount)
        elif release_time2 > release_time1:
            trim_amount = release_time2 - release_time1
            clip2 = clip2.subclip(trim_amount)

        # Ensure both clips end at the same time to avoid black frames
        min_duration = min(clip1.duration, clip2.duration)
        clip1 = clip1.set_duration(min_duration)
        clip2 = clip2.set_duration(min_duration)

        print("Rendering final overlay...")
        # Stack the transparent clip on top of the base clip
        final_clip = CompositeVideoClip([clip1, clip2])
        final_clip = final_clip.fx(vfx.speedx, speed)
        output_path = tempfile.mktemp(suffix="_final_overlay.mp4")

        # Render the final video
        final_clip.write_videofile(
            output_path,
            fps=60,
            codec="libx264",
            preset="ultrafast",
            threads=4,
            logger=None
        )

        print(f"Video saved to: {output_path}")

        # Display the result directly inside the Jupyter Notebook
        display(Video(
            output_path,
            embed=True,
            html_attributes='controls allowfullscreen style="width: 100%; max-height: 85vh;"'
        ))

        return output_path

    except Exception as e:
        print(f"Error: {str(e)}")

    finally:
        # Free up system memory and delete temporary files
        if clip1:
            try: clip1.close()
            except: pass
        if clip2:
            try: clip2.close()
            except: pass

        for f in [file1_raw, file2_raw, file1_clean, file2_clean]:
            if f and os.path.exists(f):
                try: os.remove(f)
                except: pass

def find_release_point(url, start_time=1.0, end_time=2.0):
    """Shows exactly 10 frames to find the release timestamp."""
    print("Fetching video and generating 10 frames...")
    # Fetch, download, and normalize the target video
    mp4_url = get_mp4_url(url)
    vid_file = download_video(mp4_url)
    clean_vid_file = normalize_video(vid_file)
    clip = VideoFileClip(clean_vid_file)

    # Generate 10 evenly spaced timestamps within the requested window
    times = np.linspace(start_time, end_time, 10)

    # Set up a 2x5 visual grid using Matplotlib
    fig, axes = plt.subplots(2, 5, figsize=(18, 7))
    fig.suptitle(f"Frame Inspector: {start_time}s to {end_time}s", fontsize=16)
    axes_flat = axes.flatten()

    # Extract and display the frame for each timestamp
    for i, ax in enumerate(axes_flat):
        t = times[i]
        try:
            frame = clip.get_frame(t)
            ax.imshow(frame)
            ax.set_title(f"Time: {t:.3f}s", fontsize=12, fontweight='bold')
        except Exception:
            # Handles edge cases where the requested time exceeds video length
            ax.set_title("End of Video")
        ax.axis("off") # Hide gridlines/axes for a cleaner image

    plt.tight_layout()
    plt.show()

    # Cleanup memory and temp files
    clip.close()
    for f in [vid_file, clean_vid_file]:
        if os.path.exists(f):
            try: os.remove(f)
            except: pass

/usr/local/lib/python3.13/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.13/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.13/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.13/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':

  from pkg_resources import resource_stream, resource_exists



In [28]:
#Example Usage (Generation of Stewart Overlay):

# pitch_1_url = "https://baseballsavant.mlb.com/sporty-videos?playId=f6097413-9934-396d-9c4d-638fd3bd7c47"
# pitch_2_url = "https://baseballsavant.mlb.com/sporty-videos?playId=e5d0db26-d797-328c-b249-7e1d93069389"

# Step 1: Use find_release_point to locate the exact frame the ball leaves the hand.
# Start between 2.5 and 3.5 seconds, then narrow the time interval until you clearly see the release

# find_release_point(pitch_1_url, start_time=3.0, end_time=3.05)
# Output: 10 frames from video allowing you to see that pitch 1 is released at 3.033s

# Do the same thing for pitch 2
# find_release_point(pitch_2_url, start_time=2.95, end_time=3.0)
# Output: 10 frames from video allowing you to see that pitch 2 is released at 3.033s

# Step 2: Put these values into the create_overlay function
# final_file = create_overlay(url1=pitch_1_url, release_time1=3.033, url2=pitch_2_url, release_time2=3.00, speed=1)

# Output is an overlay of the two pitches with a normal playback speed
# Can get a slo-mo overlay by changing speed to 0.5.